# SQL warehouse connectivity check

Purpose: prove the `databricks-sql-connector` path works with zero Spark.
Run each cell ONCE and stop. Every execution bills Serverless quota,
so do not loop, retry aggressively, or leave anything polling.

Auth is env-only (repo `.env`): `DATABRICKS_HOST`, `DATABRICKS_HTTP_PATH`,
`DATABRICKS_TOKEN`. No local profiles.

Note: a stopped warehouse auto-resumes on first query (1-2 min). That wait
is normal, not a hang.

In [38]:
import os
from dotenv import load_dotenv
load_dotenv()

from databricks import sql

import polars as pl


pl.Config.set_fmt_str_lengths(1000)

polars.config.Config

In [2]:

HOST = os.environ["DATABRICKS_HOST"].replace("https://", "")
HTTP_PATH = os.getenv("DATABRICKS_HTTP_PATH", "/sql/1.0/warehouses/ffd32e8cc089ac87")

conn = sql.connect(
    server_hostname=HOST,
    http_path=HTTP_PATH,
    access_token=os.environ["DATABRICKS_TOKEN"],
)
cur = conn.cursor()
print("connected")

connected


In [3]:
# 1. Tables in the schema.
cur.execute("SHOW TABLES IN workspace.btc_data")
for r in cur.fetchall():
    print(r)

Row(database='btc_data', tableName='blocks', isTemporary=False)
Row(database='btc_data', tableName='blocks_source', isTemporary=False)
Row(database='btc_data', tableName='gold_daily_fee_metrics', isTemporary=False)
Row(database='btc_data', tableName='price_minute', isTemporary=False)
Row(database='btc_data', tableName='silver_transactions_flat', isTemporary=False)
Row(database='btc_data', tableName='test_blocks_autoload', isTemporary=False)
Row(database='btc_data', tableName='transactions', isTemporary=False)
Row(database='btc_data', tableName='tx_source', isTemporary=False)


In [10]:
from polars.dataframe.frame import DataFrame
from polars.series.series import Series
tbl = cur.execute("""
select compute,  statement_type, end_time - start_time as duration, start_time, read_rows,read_partitions,
read_files_bytes/1000 as bytes_read_mb
from system.query.history


"""

).fetchall_arrow()


In [16]:
# group by start_time as date and count all rows and sum read_rows and read_partitions and bytes_read_mb
df: DataFrame | Series = pl.from_arrow(tbl)
df = df.with_columns(pl.col("start_time").cast(pl.Datetime).dt.date().alias("date"))
df.group_by("date").agg([
    pl.count("compute").alias("query_count"),
    pl.sum("read_rows").alias("total_read_rows"),
    pl.sum("read_partitions").alias("total_read_partitions"),
    pl.sum("bytes_read_mb").alias("total_bytes_read_mb"),
]).sort("date")['query_count'].describe()



statistic,value
str,f64
"""count""",80.0
"""null_count""",0.0
"""mean""",592.5875
"""std""",894.898901
"""min""",1.0
"""25%""",147.0
"""50%""",516.0
"""75%""",855.0
"""max""",7877.0


In [14]:
# plot histogram of qquery_count by date    
df.group_by("date").agg([
    pl.count("compute").alias("query_count"),
]).sort("date").plot.bar(x="date", y="query_count")

alt.Chart(...)

In [ ]:
query = """
select * 

from 
    system.billing.usage
where 
    usage_date >= CURRENT_DATE()  - INTERVAL 30 DAYS


"""

tbl = cur.execute(query).fetchall_arrow()


CloudFetch download slower than threshold: 0.0869 MB/s (threshold: 0.1 MB/s) from https://us-east-2.storage.cloud.databricks.com/api/2.0/fs/files/WorkspaceInternal/Jobs/sql/2026-10-23/00/results_2026-10-23T00:58:13Z_66f1efd8-6398-4f24-acd6-c79323e79ede?v=1&workspaceId=7474655817002792&wkt=cp&X-Databricks-Consumer-Network-Zone=internet&sn=ohio-prod&X-Databricks-Key-Version=01f17f11-d88d-12f7-be05-75e9965cc1a5&X-Databricks-Issued=20261023T005814Z&X-Databricks-TTL=899986&X-Databricks-Signature=Aa6vWOBQYOO946dVyL-cx63BA_-qAzoMNloBsRiTnBmwwQ-w8w


usage_date,total_usage_quantity,total_sku_count
date,"decimal[38,18]",u32
2026-10-08,51.789457201027694583,815
2026-10-17,51.627883110867813358,822
2026-09-26,52.836544994970308118,824
2026-10-20,56.215472877481531903,821
2026-09-23,48.261431862836970102,815
…,…,…
2026-09-25,49.870522829263054473,795
2026-10-07,50.463910079007769967,768
2026-10-16,39.738055682130538191,784


In [27]:
df = pl.from_arrow(tbl)


cols = ["sku_name","cloud","usage_date","usage_start_time",
        "usage_end_time","usage_unit","usage_quantity","record_type","usage_type"]


df[cols].group_by("usage_date").agg([
    pl.sum("usage_quantity").round().alias("total_usage_quantity").cast(pl.Int64),
    pl.count("sku_name").alias("total_sku_count")])['total_usage_quantity'].describe()

statistic,value
str,f64
"""count""",27.0
"""null_count""",0.0
"""mean""",49.296296
"""std""",13.617903
"""min""",6.0
"""25%""",49.0
"""50%""",51.0
"""75%""",53.0
"""max""",95.0


In [28]:

df[cols].group_by("usage_date").agg([
    pl.sum("usage_quantity").round().alias("total_usage_quantity").cast(pl.Int64),
    pl.count("sku_name").alias("total_sku_count")]).plot.bar(x="usage_date", y="total_usage_quantity")

alt.Chart(...)

In [49]:
# usage by 
df[cols]

sku_name,cloud,usage_date,usage_start_time,usage_end_time,usage_unit,usage_quantity,record_type,usage_type
str,str,date,"datetime[μs, Etc/UTC]","datetime[μs, Etc/UTC]",str,"decimal[38,18]",str,str
"""PREMIUM_JOBS_SERVERLESS_COMPUTE_US_EAST_OHIO""","""AWS""",2026-09-23,2026-09-23 00:10:00 UTC,2026-09-23 00:20:00 UTC,"""DBU""",0.277364000000000000,"""ORIGINAL""","""COMPUTE_TIME"""
"""PREMIUM_JOBS_SERVERLESS_COMPUTE_US_EAST_OHIO""","""AWS""",2026-09-23,2026-09-23 00:00:00 UTC,2026-09-23 00:10:00 UTC,"""DBU""",0.291501714285714286,"""ORIGINAL""","""COMPUTE_TIME"""
"""PREMIUM_JOBS_SERVERLESS_COMPUTE_US_EAST_OHIO""","""AWS""",2026-09-23,2026-09-23 00:20:00 UTC,2026-09-23 00:30:00 UTC,"""DBU""",0.398163999999999981,"""ORIGINAL""","""COMPUTE_TIME"""
"""PREMIUM_JOBS_SERVERLESS_COMPUTE_US_EAST_OHIO""","""AWS""",2026-09-23,2026-09-23 00:00:00 UTC,2026-09-23 01:00:00 UTC,"""DBU""",0.006663300000000000,"""ORIGINAL""","""COMPUTE_TIME"""
"""PREMIUM_JOBS_SERVERLESS_COMPUTE_US_EAST_OHIO""","""AWS""",2026-09-23,2026-09-23 00:00:00 UTC,2026-09-23 01:00:00 UTC,"""DBU""",0.007821975000000000,"""ORIGINAL""","""COMPUTE_TIME"""
…,…,…,…,…,…,…,…,…
"""PREMIUM_DATABRICKS_STORAGE_US_EAST_OHIO""","""AWS""",2026-10-22,2026-10-22 10:00:00 UTC,2026-10-22 11:00:00 UTC,"""DSU""",0.190225000000000000,"""ORIGINAL""","""API_OPERATION"""
"""PREMIUM_DATABRICKS_STORAGE_US_EAST_OHIO""","""AWS""",2026-10-22,2026-10-22 11:00:00 UTC,2026-10-22 12:00:00 UTC,"""DSU""",0.065437400000000000,"""ORIGINAL""","""API_OPERATION"""
"""PREMIUM_DATABRICKS_STORAGE_US_EAST_OHIO""","""AWS""",2026-10-22,2026-10-22 11:00:00 UTC,2026-10-22 12:00:00 UTC,"""DSU""",0.158966400000000000,"""ORIGINAL""","""API_OPERATION"""


In [48]:
df[cols]['sku_name'].value_counts().sort(by ="count", descending=True)

sku_name,count
str,u32
"""PREMIUM_JOBS_SERVERLESS_COMPUTE_US_EAST_OHIO""",7309
"""PUBLIC_CONNECTIVITY_DATA_PROCESSED_US_EAST_OHIO""",6973
"""INTERNET_EGRESS_FROM_US_EAST_OHIO""",3487
"""PREMIUM_DATABRICKS_STORAGE_US_EAST_OHIO""",2350
"""PREMIUM_ALL_PURPOSE_SERVERLESS_COMPUTE_US_EAST_OHIO""",274
"""GENIE_FREE_USAGE""",16
"""PREMIUM_SERVERLESS_SQL_COMPUTE_US_EAST_OHIO""",7


In [42]:
# 3. Chain + price tips. Partition-filtered, no full scans.
cur.execute(
    "SELECT max(number) AS tip FROM workspace.btc_data.blocks "
    "WHERE date >= current_date - 7"
)
print("blocks tip:", cur.fetchall())
cur.execute(
    "SELECT max(candle_start_ts) AS tip FROM workspace.btc_data.price_minute "
    "WHERE date >= current_date - 30"
)
print("price tip:", cur.fetchall())

blocks tip: [Row(tip=972680)]
price tip: [Row(tip=datetime.datetime(2026, 10, 23, 1, 0, tzinfo=<StaticTzInfo 'Etc/UTC'>))]


In [ ]:
# 4. Always close the session when done testing.
cur.close()
conn.close()
print("closed - check query history: expect a handful of warehouse queries, no Spark sessions")